## Test the greedy layout design strategy

In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *

result_dir = f"result/layout/greedy/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

package = Package((50, 50), FabricateModel(20, 10), design_strategy="greedy", result_dir=result_dir)
# package = Package(design_strategy="greedy", result_dir=result_dir)

package.save_design_history()
package.save()
print("Package score:", package.estimate_score())

Initial score:  33.878021187989866
New action:  (ComputeModule_1_1_48_48, Strip, Y, 4)
New score:  80.83825999419155
New action:  (ComputeModule_31_1_34_48, Strip, X, 20)
New score:  104.43789658919748
Package score:  86.76993159278022


In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.element.package import *

result_dir = f"result/layout/greedy/example_20250617111701"
package = Package(pre_designed_package_path=os.path.join(result_dir, "package.pkl"))
print("Package score:", package.estimate_score())

## Test the rl-based layout design strategy

In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.layout.env import QuNoCEnv
from qunoc.layout.agent import LayoutAgent

result_dir = f"result/layout/rl/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

env = QuNoCEnv((50, 50), FabricateModel(20, 10), result_dir=result_dir)
# env = QuNoCEnv(result_dir=result_dir)
agent = LayoutAgent(env, result_dir=result_dir)
agent.train(1000)
package = agent.design()

package.save_design_history()
package.save()
print("Package score:", package.estimate_score())

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Training agent...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 16.9     |
|    ep_rew_mean     | -1.4e+09 |
| time/              |          |
|    fps             | 8        |
|    iterations      | 1        |
|    time_elapsed    | 249      |
|    total_timesteps | 2048     |
---------------------------------
Designing layout...
Initial score:  39.61728499372216
New action:  [17 40  2  1  1]
New score:  74.54653705609819
New action:  [28 23  1  0  3]
New score:  65.51810600817757
New action:  [28 23  1  0  3]
New score:  71.03631243430449
New action:  [28 23  1  0  3]
New score:  70.1210073894442
New action:  [28 23  1  0  3]
New score:  80.50475805612783
New action:  [28 23  1  0  3]
New score:  70.70797684069667
New action:  [28 23  1  0  3]
New score:  68.75900348327541
New action:  [28 23  1  0  3]
New score:  67.74360363335184
New action:  [2

## Test the benchmark generation

In [1]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.element.benchmark import *

vqe_program = vqe(5)
print(vqe_program)
qft_program = qft(5)
print(qft_program)
qaoa_program = qaoa(5)
print(qaoa_program)
bv_program = bv(5)
print(bv_program)

vqe program: Control_0_1, Control_0_2, Control_0_3, Control_0_4, Control_1_2, Control_1_3, Control_1_4, Control_2_3, Control_2_4, Control_3_4
qft program: Control_1_0, Control_2_0, Control_3_0, Control_2_1, Control_4_0, Control_3_1, Control_4_1, Control_3_2, Control_4_2, Control_4_3
qaoa program: Control_2_4, Control_3_4, Control_0_2, Control_1_4, Control_1_2
bv program: Control_0_3, Control_0_4


## Test the routing algorithm

In [1]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.INFO)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.element.benchmark import *

algo = "vqe"
data_qubit_num = 5

# result_dir = f"result/routing/example_{get_timestamp()}"
result_dir = f"result/routing/example_{algo}_{data_qubit_num}"
os.makedirs(result_dir, exist_ok=True)

package = Package(package_size=(9, 9), fab_model=FabricateModel(9, 9), design_strategy="manual", result_dir=result_dir)
package.design_module_top_down(package.compute_modules[0], DesignTemplate.Grid, DesignOrientation.X, 3)

package.save_design_history()
package.save()
print("Package score:", package.estimate_score(), "\n")

program = get_program(algo, data_qubit_num)
gate_depend_fig = program.draw_depend()
gate_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_gate_depend.jpg"), bbox_inches='tight')
qiskit_circuit_fig = program.draw_circuit()
qiskit_circuit_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qiskit_circuit.jpg"), bbox_inches='tight')

Package score: 77.10728265692612 



In [2]:
from qunoc.routing.router import *

router = Router(package=package, program=program, result_dir=result_dir)
router.map_qubits()

INFO:root:Initial qubit mapping: {0: (2, 2), 1: (1, 6), 2: (2, 5), 3: (6, 2), 4: (6, 5)}


In [3]:
router.compile_blocks_and_actions()
router.save_block_history()
block_depend_fig = router.draw_block_depend()
block_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_block_depend.jpg"), bbox_inches='tight')

INFO:root:--------------------------------------------------
INFO:root:Constructed TeleBlock 0: Teleport qubit 0 to [1, 2, 3, 4].
INFO:root:Transformed GhzPreparation 0: Prepare 5-GHZ state for logic qubits [0, 1, 2, 3, 4] on physical qubits [(7, 4), (8, 4), (5, 4), (6, 4), (8, 3)].
INFO:root:Transformed SwapGate 1: Route logic qubit 0 with swap gate between physical qubits (2, 2) and (2, 1).
INFO:root:Transformed Transport 2: Transport logic qubit 5 from physical qubit (7, 4) to physical qubit (4, 4).
INFO:root:Transformed Interchange 3: Interchange logic qubit 5 between physical qubits ((4, 4), (4, 3)).
INFO:root:Transformed Transport 4: Transport logic qubit 5 from physical qubit (4, 3) to physical qubit (4, 1).
INFO:root:Transformed Alight 5: Alight logic qubit 5 from physical qubit (4, 1) to physical qubit (3, 1).
INFO:root:Transformed Teleport 6: Teleport data qubit 0 with ancilla qubit 5 on physical qubit (3, 1).
INFO:root:Transformed SwapGate 7: Route logic qubit 1 with swap ga

In [4]:
router.schedule_actions_to_circuit()
router.save_action_history()
action_depend_fig = router.draw_action_depend()
action_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_action_depend.jpg"), bbox_inches='tight')
circuit = router.circuit
circuit.to_pickle(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qunoc_circuit.pkl"))
print("-" * 50)
print(f"Program {program.algo_name}-{program.data_qubit_num} routing result: QST_Fidelity_{router.estimate_average_qst_fide()}, Depth_{router.estimate_depth()}")

INFO:root:Updated no depend actions: {0, 1, 7, 18}.
INFO:root:--------------------------------------------------
INFO:root:Extended circuit with BoardingStage 0.
INFO:root:Current qubit mapping: {0: (2, 2), 1: (1, 6), 2: (2, 5), 3: (6, 2), 4: (6, 5)}
INFO:root:Added GhzPreparation 0.
INFO:root:Updated no depend actions: {1, 2, 7, 9, 14, 18, 19, 22}.
INFO:root:--------------------------------------------------
INFO:root:Extended circuit with RidingStage 1.
INFO:root:Current qubit mapping: {0: (2, 2), 1: (1, 6), 2: (2, 5), 3: (6, 2), 4: (6, 5), 5: (7, 4), 6: (8, 4), 7: (5, 4), 8: (6, 4), 9: (8, 3)}
INFO:root:Added Transport 2.
INFO:root:Added Transport 9.
INFO:root:Added Transport 22.
INFO:root:Updated no depend actions: {1, 3, 7, 10, 14, 18, 19, 23}.
INFO:root:Swapping 0 and -1.
INFO:root:Added SwapGate 1.
INFO:root:Swapping 1 and -1.
INFO:root:Added SwapGate 7.
INFO:root:Swapping 3 and -1.
INFO:root:Added SwapGate 18.
INFO:root:Updated no depend actions: {3, 8, 10, 14, 19, 23}.
INFO:ro

--------------------------------------------------
Program vqe-5 routing result: QST_Fidelity_None, Depth_0.016


In [1]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.INFO)

from qunoc.element.package import *
from qunoc.routing.circuit import *

circuit = Circuit(10, dict(), dict(), nx.DiGraph(), OperationConfig())
circuit.to_pickle("test_qunoc_circuit.pkl")
circuit = Circuit.from_pickle("test_qunoc_circuit.pkl")
print(circuit)